In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim   #optimizer
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, random_split, Subset
import os

#this is how you use your own data in google drive
from google.colab import drive


#from CS 2503 tutorial
#from google.colab import output
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import glob
from PIL import Image
from torch.utils.data import Dataset

#define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1 . Mount the Google Drive
drive.mount('/content/drive')

# 2. path for saving the weights
weights_path = "/content/drive/MyDrive/FinalProject_CSC2503/resnext18_GRADCAM_weights.pth"



# 2. Define path to dataset
data_dir = "/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification Resized"



## HELPER FUNCTIONS FOR DATALOADER

### obtain all images within the 'train' or 'val' subfolder of the  fracture type folder ###
def build_image_list(root, fracture_classes, split): #root= dataset path, split = 'train' or 'val'
  items = [] # holds (image_path, label_index) tupels
  for idx, cls in enumerate(fracture_classes): #idx is integer label assigned to the class in fracture_classes list; cls is class name string ("Avulsion fracture")
    folder = os.path.join(root, cls, split) # creates custom path bc root = "/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification", cls = "Avulsion fracture", split = 'train' or 'val'
    if not os.path.isdir(folder):
      raise FileNotFoundError(f"Expected {folder} to exist")
    # if we do have a folder path, continue on looping thru images
    for ext in('*.png', '*.jpg', '*.jpeg'):
      for p in glob.glob(os.path.join(folder, ext)): # finds all matching file paths (p) in this folder
        items.append((p, idx)) # append file path p and numerical class integer label idx to the list items

  # return looks like this: ("/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification/Avulsion fracture/train/image_123.jpg", 0) -> 0 is the numerical integer idx for 'Avulsion fracture' folder
  return items # has the path and numerical label for the requested split - 'train' or 'val'

###   converts to RGB, APPLYS TRANSFORMS AND RETURNS A TUPLE ###
class ImageListDataset(Dataset):
  def __init__(self, items, transform=None):
    self.items = items # items is list return by build_image_list
    self.transform = transform
  def __len__(self):
    return len(self.items) # return # of images we have
  def __getitem__(self, idx):
    path, label = self.items[idx] # retirieve the tuple (path, label) from items list, label is numerical integer label
                                  # path is specific: /content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification/Avulsion fracture/train/img_045.jpg
    img = Image.open(path).convert('RGB') # convert to RGB bc most pretrained CNNs expect 3 channel input otherwise we'd get mismatch error
                                          # if we were training from scratch then we dont need to convert to RGB
    # apply transforms here
    if self.transform:
      img = self.transform(img)

    # img is torch.Tensor (3x224x224), label is int (0-9), path is str to the img loc ie./.../Avulsion fracture/train/img_001.jpg for saving gradCAM overlay
    return img, label, path # path so we can save GRAD-CAM overlays next to source images


####################################################################



# fracture transforms for train and eval
img_size = 224
transform = transforms.Compose([
     transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 10 classes of fracture here
fracture_classes = ['Avulsion fracture',
               'Comminuted fracture',
               'Fracture Dislocation',
               'Greenstick fracture',
               'Hairline Fracture',
               'Impacted fracture',
               'Longitudinal fracture',
               'Oblique fracture',
               'Pathological fracture',
               'Spiral Fracture']

# Load the entire fracture dataset (images + labels)
fracture_dataset = torchvision.datasets.ImageFolder(root=data_dir, transform=transform)
print("Class mapping:", fracture_dataset.class_to_idx)

# Force its class_to_idx mapping to match fracture_classes defined above
fracture_dataset.class_to_idx = {cls: i for i, cls in enumerate(fracture_classes)}
fracture_dataset.classes = fracture_classes

# Wrap in DataLoader
#sdd_loader = DataLoader(sdd_dataset, batch_size=8, shuffle=False)

# Training parameters
num_classes = 10 # 10 types of fractures to classify
batch_size = 16
epochs = 20



### Create the datasets with DataLoader ###

# build_image_list func returns tuple (image_path, numerical_label(0-9))
# need to get tuples for 'train' and 'val' subfolders for ALL fracture classes
train_items = build_image_list(data_dir, fracture_classes, split='Train')
val_items = build_image_list(data_dir, fracture_classes, split='Val')
test_items = build_image_list(data_dir, fracture_classes, split='Test')

# pass items list thru ImageListDataset class and convert to RGB, apply transforms
train_dataset = ImageListDataset(train_items, transform = transform)
val_dataset = ImageListDataset(val_items, transform = transform)
test_dataset = ImageListDataset(test_items, transform = transform)

# create the DataLoader - wraps Dataset and provides mini-batchs to model during trainnig/val
# Each batch gives:
#img: the input images (as tensors)
#label: the integer class labels (e.g., 0–9 for 10 fracture types)
#path: the file path to each image (used later for overlaying Grad-CAMs)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers = 2, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers = 2, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers = 2, pin_memory = True)


# print # images for train and val sets
print("Train samples:", len(train_dataset), "Val samples:", len(val_dataset), "Test samples:", len(test_dataset))





# Load the saved weights
#resnext50.load_state_dict(torch.load(weights_path))
#resnext50.eval()  # set to evaluation mode


########### TRAINING SETUP NOW ###############



# TRAINING LOOP - 1 EPOCH - ALL EPOCHS ACCOUNTED FOR IN FOR-LOOP AFTER THIS
def training(model, DataLoaderBatch, optimizer, lossFunction, device):
  model.train()
  # track running variables
  running_loss = 0.0
  running_corrects = 0
  total = 0


  # dataLoader returns inputs (acc image tensor), labels (0-9 for 'advulsion frac') and path (loc of acc img) [we do _ bc we dont care for the path rn]
  for inputs, labels, _ in DataLoaderBatch:
    inputs = inputs.to(device)
    labels = labels.to(device)

    optimizer.zero_grad() # clears gradients from prev iteration
    outputs = model(inputs) # forward pass thru model - input image thru resnext then output is predicted scores per class
    loss = lossFunction(outputs, labels) # calc how wrong the model is w pred
    loss.backward() # backproagatiton: compute grad w/r/t every model parameter
    optimizer.step() # updates parameters w gradients just computed
    _, preds = torch.max(outputs, 1) # returns maxScorePerSample and index(predictedClass) - we only care about index(predictedClass) so _ the first value
    running_loss += loss.item() * inputs.size(0)
    running_corrects += torch.sum(preds == labels).sum().item()
    total += inputs.size(0)
  avg_loss = running_loss/total
  avg_acc = running_corrects/total

  return avg_loss, avg_acc

# EVALUATION ON THE 'VAL' FOLDERS OF THE DATASET
def evaluation(model, loader, lossFunction, device):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    with torch.no_grad(): # DO NOT UPDATE WEIGHTS OR COMPUTE GRADIENTS HERE
        for inputs, labels, _ in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = lossFunction(outputs, labels)
            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == labels).sum().item()
            total += inputs.size(0)
    return running_loss/total, running_corrects/total



# TEST FUNCTION
def test(model, loader, lossFunction, device):
  model.eval()
  correct = 0
  total = 0
  total_loss = 0.0

  with torch.no_grad():
    for images, labels, paths in loader:
      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)
      loss = lossFunction(outputs, labels)

      total_loss += loss.item()*images.size(0)

      predictions = outputs.argmax(dim=1)
      correct += (predictions == labels).sum().item()
      total += images.size(0)


  avg_loss = total_loss/len(loader.dataset)
  test_acc = correct /len(loader.dataset)

  print(f"Test Loss: {avg_loss:.4f} | Test Accuracy: {test_acc:.4f}")

  #return avg_loss, accuracy




### Load the pretrained resneXt50 model ###
resnet18 = models.resnet18(pretrained=True)
resnet18.fc = nn.Linear(resnet18.fc.in_features, num_classes)
resnext50 = resnet18.to(device)

# loss and training setup
lossFunction = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnext50.parameters(), lr=0.0001)



####### CHECK IF WEIGHTS WERE SAVED
start_epoch = 0
best_val_acc = 0.0

# Optionally load weights if continuing training or evaluating
if os.path.exists(weights_path):

  #state = torch.load(weights_path, map_location=device)

  checkpoint = torch.load(weights_path, map_location=device)
  resnext50.load_state_dict(checkpoint['model_state_dict'])
  optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
  start_epoch = checkpoint['epoch'] + 1 # resume from next epoch
  best_val_acc = checkpoint['val_acc']
  print(f"✅ Loaded checkpoint from epoch {start_epoch}")
  print("Models loaded successfully!")

else:
  print("NO previously saved models found!")



# RUN THE TRAINING AND EVALUATION FOR THE NUMBER OF SPECIFIED EPOCHS NOW
#best_val_acc = 0.0


# BEGIN LOOPING THRU EPOCHS
for epoch in range(start_epoch, epochs):
    train_loss, train_acc = training(resnext50, train_loader, optimizer, lossFunction, device)
    val_loss, val_acc = evaluation(resnext50, val_loader, lossFunction, device)
    #scheduler.step()
    print(f"Epoch {epoch+1}/{epochs}  Train loss: {train_loss:.4f} acc: {train_acc:.4f}  Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    # save checkpoint
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        # saving epochs, optimizer, state
        torch.save({
            'epoch': epoch,
            'model_state_dict': resnext50.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc
        }, weights_path)
        print(f"✅ Saved best model (epoch {epoch+1}) with val_acc={val_acc:.4f} to {weights_path}")



print("MUST LOAD best saved model before testing: ")

checkpoint = torch.load(weights_path, map_location=device)
resnext50.load_state_dict(checkpoint["model_state_dict"])
#linear_head.load_state_dict(checkpoint["linear_state_dict"])

print(f"Loaded best model from epoch {checkpoint['epoch']+1} with val_acc={checkpoint['val_acc']:.4f}")


# Run the TEST FUNCTION ONCE TRAINING AND EVALUATION ARE DONE
test(resnext50, test_loader, lossFunction, device)





################ IMPLEMENT GRAD-CAM AFTER TRAINING ##############

# Save the Grad-CAM images to this path
save_gradcam_dir = "/content/drive/MyDrive/FinalProject_CSC2503/GradCAM_Images"

# Need to "hook" the last conv layers activations and gradients so we can generate the heat maps

activations = None # activations = the (A_ij)^k tensor feature maps where k = feature maps, ij are the pixels
gradients = None # gradients = partial_derivative(y^c)/partial_derivative((Aij)^k)



# THEORY: these are the (A_ij)^k values  used in the final sum
# attach forward_hook to a conv layer; after a fwd pass, conv layers output (activations tensor above) is passed to this hook function (IN OUR CASE, WE ONLY PASS THE LAST CONV LAYER HERE)
def forward_hook(module, input, output): # this is the convention for a hook, even tho we only use "output" here
    # Save the activation maps (the "A^k")
    global activations
    activations = output.detach()  # shape [B = batch_size, K=num channels, H, W = spatial dimensions, ie 7x7], copy output to activations variable, .detach so their gradients aren't updated anymore


# THEORY: gradients are the partial_derivative(y^c)/partial_derivative((Aij)^k) values used to compute the channel weights α_k
# attach backward_hook on activation tensor
def backward_hook(grad):
    # Save gradients w.r.t. the activation maps (the dY/dA)
    global gradients
    gradients = grad.detach()      # shape [B=batch_size, K=num_channels, H, W]

# Attach forward hook to target layer (last conv layer (-1)) resnext50.layer4[-1].conv3 or model.layer4[-1]
# resnext50 is the model here
target_layer = resnext50.layer4[-1]   # you can try .conv3 if you want the conv specifically

# this line is only so we can remove the hook at the end
#PYTORCH calls the function inside the FORWARD PASS of the model whenever tht layer produces an output
# output is the activation tensor produced by that conv layer in the curr forward pass
handle_forward = target_layer.register_forward_hook(forward_hook)
# adding backward_hook registration here
handle_backward = target_layer.register_full_backward_hook(lambda module, grad_in, grad_out: backward_hook(grad_out[0]))

##### Function compute Grad-CAM for a single image tensor #####
### THIS FUNC MUST BE CALLED IN THE FOR LOOP AFTER WHEN WE ARE OVERLAYING THE GRAD-CAM MAPS
def compute_gradcam(model, input_tensor, target_class=None):
    """
    input_tensor: shape [1, 3, H, W] preprocessed (normalized)
    target_class: integer class index to compute Grad-CAM for. If None, uses predicted class.
    returns: cam_upsampled (HxW numpy), activations, gradients
    """
    model.eval()
    #CLEAR activations and gradients variables here before generating maps
    global activations, gradients
    activations = None
    gradients = None

    input_tensor = input_tensor.to(device)
    input_tensor.requires_grad = True

    # START OUR FORWARD PASS HERE
    # forward_hook runs automatically in this line bc its called by register_forward_hook() function and activations are saved in the global variable  after the conv layers
    outputs = model(input_tensor)


    # GRAD-CAM uses the gradients of the class score (y^c) for target class
    # if no target_sclass is given (ie. "Avulsion fracture"), then choose one
    if target_class is None:
        pred = outputs.argmax(dim=1).item() # outputs is shape [1, num_classes] for a single img
        target_class = pred # use the predicted class if a class isnt specified in functino signature

    # scalar score for class c for the single image in the batch
    score = outputs[0, target_class] # this gets the scalar y^c (models raw score for the class 'avulsion fracture')
    model.zero_grad() # clear gradients before backprop

    # .backward() starts back propagation
    #backward_hook func automatically stores the grad of the class score wrt the activations (ONLY IF IT REGISTERED W .register_full_backward_hook BUT SINCE ITS NOT, THE BACKWARD_HOKO FUNC IS NEVER USED)
    score.backward(retain_graph=True)   # backprop the scalar score bc GRAD-CAM uses the derivative of the class score w/r/t activations: partial_deriv(y^c)/partial_derivative(Aij^k)


    # if backward_hook isn't set; get gradient by registering on activations:
    if gradients is None:
        # activations variable is a tensor; register hook on it and recompute backward:
        # backward hook used on activation's grad
        grads = activations.grad



    # Now compute channel-wise weights: global average pooling of gradients over spatial dims
    grads = gradients   # [1, K, H, W]
    # calculating α_k^c​ which is the avg gradient over sptial locations for channel k, class c
    weights = grads.mean(dim=(2,3), keepdim=True)   # [1, K, 1, 1]


    # Weighted sum of activations

    #multiply each channel map Aij^k by scalar weight α_k and sum accross channels for a class
    cam = (weights * activations).sum(dim=1, keepdim=True)  # [1, 1, H, W]
    cam = F.relu(cam)   # keep only positive sums with ReLU
    # interpolate the smaller spatial dims to larger ones so we can overlay the GRAD-CAM map
    cam = F.interpolate(cam, size=(img_size, img_size), mode='bilinear', align_corners=False)  # upsample to input size
    cam = cam.squeeze().cpu().numpy() # get 2D array here

    #normalize to [0, 1] so we can map values to colours
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    return cam, target_class


##### Create overlays for the entire validation set and save images #####

#LOOPING THRU ALL THE VALIDATION IMAGES AGAIN HERE TO CREATE THE OVERLAY
resnext50.eval()
# we must detach forward hook result already; but need gradients -> so don't wrap in torch.no_grad
for inputs, labels, paths in val_loader: #val_loader returns batches of 16, path is the loc saved at, labels are ground-truth integer labels, inputs are tensor[B, 3, H, W]

    # PROCESS ONE IMG AT A TIME FROM THE BATCH from val_loader
    for i in range(inputs.size(0)):

        img_tensor = inputs[i:i+1]  # [1,3,H,W] # somehow keeps batch dim = 1
        orig_path = paths[i]

        # SEND TO COMPUTE_GRADCAM FUNCTION - runs forward, backward for tht single image
        # cam = 2D array (heatmap 0-1)
        cam, target_class = compute_gradcam(resnext50, img_tensor, target_class=None)

        # LABEL NAMES
        gt_label_name = fracture_classes[labels[i].item()]
        pred_label_name = fracture_classes[target_class]


        # load original image (un-normalized) to overlay
        orig = Image.open(orig_path).convert('RGB').resize((img_size, img_size))

        # convert cam to heatmap (RGB) - GET COLOURS HERE
        cmap = plt.get_cmap('jet')
        heatmap = cmap(cam)[:, :, :3]
        heatmap_img = Image.fromarray((heatmap * 255).astype(np.uint8))

        # overlay (alpha composite)
        overlay = Image.blend(orig, heatmap_img, alpha=0.4) # blends 40% heatmap, 60% orig image


        # DISPLAY THE GRAD-CAM OVERLAYS
        plt.figure(figsize=(5, 5))
        plt.imshow(overlay)
        plt.title(f"Original GT: {gt_label_name}\nPredicted class: {pred_label_name}")
        plt.axis("off")
        plt.show()


        out_name = os.path.basename(orig_path)
        out_path = os.path.join(save_gradcam_dir, f"{target_class}_{out_name}")
        overlay.save(out_path)
        print("Saved", out_path)

# Remove forward hook at the end
handle_forward.remove()
handle_backward.remove()




